# TP 1 — Du texte brut au bigramme

On part de Tiny Shakespeare et on apprend une table : après un caractère, quels caractères sont plausibles. Ça n'écrit pas de vraies phrases. Le but est de voir comment on fabrique les exemples, et ce que l'entraînement change dans les poids.



In [1]:
import os
from urllib.request import urlretrieve

import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("input.txt"):
    urlretrieve(url, "input.txt")

# utf-8 : même lecture des octets, quel que soit l'ordi
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(len(text), "caractères,", len(set(text)), "symboles distincts")
print(text[:400])


1115394 caractères, 65 symboles distincts
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it 


## Étape 1 — Télécharger et regarder le corpus

Ici un token, c'est un caractère. Le vocabulaire reste petit, une soixantaine de symboles, et on pourra lire la table à la fin. On précise encoding="utf-8" pour lire les octets du fichier de la même façon sur toutes les machines, sans dépendre de l'encodage par défaut. Le fichier n'est pas déjà séparé en exemples : c'est une seule chaîne, avec les sauts de ligne et les noms de personnages dedans. Les exemples, on les fabrique ensuite en glissant une fenêtre.


In [2]:
chars = sorted(list(set(text)))  # tri : le même caractère garde le même indice
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}  # caractère -> indice
itos = {i: ch for i, ch in enumerate(chars)}  # indice -> caractère
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

message = "hii there"
print("".join(chars))
print(encode(message))
print(decode(encode(message)), "| bijection :", decode(encode(message)) == message)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there | bijection : True


## Étape 2 — Passer des caractères aux entiers

On trie chars pour que le même caractère ait toujours le même numéro. Un set n'a pas d'ordre stable d'une exécution à l'autre. Le test decode(encode(message)) == message vérifie que les deux dictionnaires sont bien l'inverse l'un de l'autre : aucun caractère n'est perdu ni fusionné. L'indice n'est pas une grandeur. 20 n'est pas « deux fois » 10, c'est juste le numéro d'une ligne dans la table.


In [3]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))  # 90 % train, la fin sert de validation
train_data, val_data = data[:n], data[n:]

block_size = 8  # T, longueur du contexte
x = train_data[:block_size]
y = train_data[1:block_size + 1]  # y = x décalé d'un caractère

print("train", len(train_data), "val", len(val_data))
for t in range(block_size):
    contexte = decode(x[:t + 1].tolist())
    cible = decode([int(y[t])])
    print(f"t={t}  {contexte!r:>12} -> {cible!r}")


train 1003854 val 111540
t=0           'F' -> 'i'
t=1          'Fi' -> 'r'
t=2         'Fir' -> 's'
t=3        'Firs' -> 't'
t=4       'First' -> ' '
t=5      'First ' -> 'C'
t=6     'First C' -> 'i'
t=7    'First Ci' -> 't'


## Étape 3 — Fabriquer les cibles

Ce bloc produit 8 prédictions, une par position. y est x décalé d'un caractère : personne n'a annoté le texte, la cible est simplement le caractère d'après. Pour t = 3, le contexte est les 4 premiers caractères (Firs au début du corpus) et la cible est le suivant (t).

On ne mélange pas train et validation. Ce sont deux morceaux collés, 90 % puis 10 %. Deux fenêtres voisines se ressemblent trop. Si on tirait la validation au milieu du train, la loss mesurerait surtout de la mémorisation.


In [4]:
batch_size = 4

def get_batch(split):
    source = train_data if split == "train" else val_data
    # on retire block_size pour que le caractère i + block_size existe encore
    starts = torch.randint(len(source) - block_size, (batch_size,))
    xb = torch.stack([source[i : i + block_size] for i in starts])
    yb = torch.stack([source[i + 1 : i + block_size + 1] for i in starts])
    return xb, yb

xb, yb = get_batch("train")
print(tuple(xb.shape), tuple(yb.shape))
print(repr(decode(xb[0].tolist())))
print(repr(decode(yb[0].tolist())))


(4, 8) (4, 8)
"Let's he"
"et's hea"


## Étape 4 — Un mini-batch

La forme est (B, T) = (4, 8). Une ligne est une séquence, une colonne une position. On retire block_size à la borne du tirage pour que le caractère juste après la fenêtre existe encore. Sinon la cible sortirait du tableau.

B = 32 mettrait 32 fenêtres au lieu de 4. Le gradient serait une moyenne sur plus d'exemples, donc moins bruité, pour plus de calcul. T = 64 allongerait chaque fenêtre. Pour ce bigramme, ça n'ajoute pas de mémoire : chaque position ne lit que son propre caractère. Ça empile juste plus d'exemples dans le batch. Au TP 4, T sera vraiment la taille du passé visible.


In [5]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # ligne i = scores du caractère qui suit i. Table C x C.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            # chaque position est un exemple de classification indépendant
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]  # seul le dernier pas décide du prochain caractère
            probs = F.softmax(logits, dim=-1)  # scores -> probabilités
            idx = torch.cat((idx, torch.multinomial(probs, num_samples=1)), dim=1)
        return idx

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print("logits", tuple(logits.shape), "loss", round(loss.item(), 3))
print("hasard ln(C)", round(torch.log(torch.tensor(float(vocab_size))).item(), 3))
print("paramètres", sum(p.numel() for p in model.parameters()))


logits (4, 8, 65) loss 5.036
hasard ln(C) 4.174
paramètres 4225


## Étape 5 — La table qui donne les scores

La table est en C × C, avec C la taille du vocabulaire. La ligne i répond à « je viens de voir le caractère i », et chaque colonne est le score du caractère suivant. Au TP 4, cette table unique se sépare en deux : l'embedding de token, de taille (vocab, n_embd), et lm_head, qui repasse de n_embd vers le vocabulaire. Entre les deux, l'attention pourra modifier le vecteur. Ici il n'y a rien entre les deux, le score ne dépend pas du passé.

On aplatit B × T seulement pour la loss. cross_entropy attend N exemples et C classes, et chaque position est un exemple. À la génération on ne garde que le dernier pas, logits[:, -1, :], parce que les caractères déjà écrits ont été choisis avant. On tire avec multinomial plutôt qu'avec argmax, sinon on répète toujours le même successeur.


In [6]:
def generer(modele, n=300):
    depart = torch.zeros((1, 1), dtype=torch.long)
    return decode(modele.generate(depart, n)[0].tolist())

@torch.no_grad()
def loss_moyenne(modele, n=30):
    total = 0.0
    for _ in range(n):
        xb_, yb_ = get_batch("train")
        _, loss_ = modele(xb_, yb_)
        total += loss_.item()
    return total / n

print("--- avant entraînement ---")
print("loss", round(loss_moyenne(model), 3))
print(generer(model))


--- avant entraînement ---
loss 4.66

kRfzWWSfHOPrFSyD-kwCqtJnH??Akn:W,NQXtBmMSmjIU3n:IjPW3eHRxp?exmOmhyCGCR&XJs$zq$OpZzYZ,ho:IG!ajRfhOmM.uAEsF-Q bwQKvsR?A$B
yrPbZ:HE
FDi&mUIe:sn3MsFhoejwkotQugaaXlTTbAtIl;WSJs:uWtrjGjtuW;tf;t&OZaOhwqXaY;rE?bfKalcn-Sig;yVezqkM3IM-c$Vlc?ThUZzEd
kZl;?epKXY.unq$dZp&$rdLBcPOxjtIYjFEUcPG?fkydevq:YOM&$r$Di'Wgg


In [7]:
# Batch de 4 et lr 1e-3, comme dans l'énoncé : 100 pas ne voient presque rien
# (4 x 8 x 100 = 3200 étiquettes pour 65 x 65 cases). On regarde, puis on
# passe à un batch de 32 et lr 1e-2. Pour une seule table, 1e-3 est lent ;
# c'est plutôt le taux du réseau profond du TP 4.
batch_size = 4
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for _ in range(100):
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

print("--- après 100 pas, batch 4, lr 1e-3 ---")
print("loss", round(loss_moyenne(model), 3))
print(generer(model))


--- après 100 pas, batch 4, lr 1e-3 ---
loss 4.583

!kWCkWlf;aZNblmqdl?ev
tIl;3n
qCF&GhRRSpkM uLLwkvsq'-MjFOr
Aqq3bwDjucPy&DimuIr zX'WtJs:LCbbpzkHMuRfkETgLojWtIvSesXgkPyN,Oxa yajIPU3bdxtV
lML
lHEU3Q F3,CKkYDUzwcQF'N-BeZImU,O!,edpBMusnGs!$OxMbN
t
lpZnEDCaI!f!xsThomnVVL ,NTE yXDjDNxY!NUPyDXyXloS
EddwjMBe,o?exVl;HEPim'SppkRyG
fPytwQXhpV-MMlc$Di$CoeFs3MG


In [8]:
torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
batch_size = 32  # 32 fenêtres par pas, le gradient est moins bruité qu'avec 4
opt = torch.optim.AdamW(model.parameters(), lr=1e-2)  # une seule table supporte un pas plus grand que 1e-3

print("pas 0  ", round(loss_moyenne(model), 3))
for step in range(1, 1001):
    xb, yb = get_batch("train")
    _, loss = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 200 == 0:
        print(f"pas {step:<4} {loss_moyenne(model):.3f}")


pas 0   4.727
pas 200  3.129


pas 400  2.692
pas 600  2.567


pas 800  2.501
pas 1000 2.488


In [9]:
print("--- après 1000 pas, batch 32, lr 1e-2 ---")
print(generer(model, 400))

poids = model.token_embedding_table.weight.detach()

def suivants(ch, k=5):
    scores = poids[stoi[ch]]
    idx = torch.topk(scores, k).indices.tolist()
    return [itos[i] for i in idx]

for ch in ["q", "t", "h", " ", ".", "\n"]:
    print(repr(ch), "->", suivants(ch))


--- après 1000 pas, batch 32, lr 1e-2 ---

I f o Y hy? tome,io:
Hed, r ENDUENROLAt ce,
JURIf'seals or o; areas: thags cMENGAN.
RI s dsaneloles.

Wot S: and;
TEr touMyyst $?
Bth y, theliserat a gsre?nd,

s ckigecun ak.
Murow$'se
D and hotNLESTWhes thesput
Cjusonaicezred; l motho ot att wst.
QGotRIETok hy sh mefaro br'sthire IN:
WAve mwayof bsulfouritr dLUMenghoS:CEN3:dee ANoimiar!

TIO, jxcacime n ONCENG ndupechinE:
S:
Wee mivens teandl CIU
'q' -> ['u', '-', ',', 'f', 'H']
't' -> ['h', ' ', 'o', 'e', 'i']
'h' -> ['e', 'a', 'i', ' ', 'o']
' ' -> ['t', 'a', 'h', 's', 'm']
'.' -> ['\n', ' ', '?', 'i', 'y']
'\n' -> ['\n', 'T', 'A', 'W', 'I']


## Étapes 6 et 7 — Générer, puis entraîner

Avant l'entraînement, la loss est autour de ln(65) ≈ 4,2 et le texte est du bruit. Après 100 pas avec un batch de 4 et lr = 1e-3, elle bouge à peine : 4 × 8 × 100 = 3200 étiquettes pour une table de 65 × 65 cases, ce n'est pas assez. Après 1000 pas avec un batch de 32 et lr = 1e-2, elle descend vers 2,5. La table a compté des couples : q est suivi de u, t souvent de h, un point souvent d'un saut de ligne. On voit des espaces et des bouts de mots, pas des phrases.

Le code de génération est le même avant et après. Si le texte change, c'est que les poids de la table ont changé. Allonger T ne rend pas ce modèle plus malin : chaque position continue de lire une seule ligne. Pour utiliser le passé, il faut l'attention du TP 2.
